# Multiparticle validation: Euclidean vs polar coordinates

The Euclidean simulation uses the local validation solver `euclidean_moving_wall_solver_v1.py`; the polar simulation uses the local validation solver `polar_solver_v2.py`.

In [ ]:
import sys
import time
import importlib
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# Use only solver/helper files stored in this validation directory. This keeps
# the notebook self-contained and avoids changing/importing shared solvers from
# other folders.
_here = Path.cwd().resolve()
_candidates = [
    _here,
    _here / "Articolo_Curvilinear" / "Section 2 Validation",
]

for _candidate in _candidates:
    if (_candidate / "euclidean_moving_wall_solver_v1.py").exists():
        VALIDATION_DIR = _candidate
        break
else:
    raise FileNotFoundError("Could not find euclidean_moving_wall_solver_v1.py")

if str(VALIDATION_DIR) not in sys.path:
    sys.path.insert(0, str(VALIDATION_DIR))

import euclidean_moving_wall_solver_v1 as esol
import polar_solver_v2 as psol

importlib.reload(esol)
importlib.reload(psol)

In [ ]:
def animate_euclidean_moving_circle_clean(
    t,
    x,
    radius,
    xA,
    omega,
    box=None,
    fps=60,
    stride=1,
    s=20,
    show_box=True,
):
    idx = np.arange(0, len(t), max(1, int(stride)), dtype=np.int64)
    xA = np.asarray(xA, dtype=float)

    if box is None:
        pad = float(radius) + float(np.max(np.abs(xA))) + 0.5
        x_min = float(np.min(x[:, :, 0]) - pad)
        x_max = float(np.max(x[:, :, 0]) + pad)
        y_min = float(np.min(x[:, :, 1]) - pad)
        y_max = float(np.max(x[:, :, 1]) + pad)
    else:
        x_min, x_max, y_min, y_max = box

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)
    ax.set_xlabel("x")
    ax.set_ylabel("y")

    if show_box and box is not None:
        ax.plot(
            [x_min, x_max, x_max, x_min, x_min],
            [y_min, y_min, y_max, y_max, y_min],
            linewidth=1.0,
        )

    scat = ax.scatter(x[idx[0], :, 0], x[idx[0], :, 1], s=s)

    theta = np.linspace(0.0, 2.0 * np.pi, 200)
    c0 = xA * np.sin(float(omega) * float(t[idx[0]]))
    wall, = ax.plot(
        c0[0] + radius * np.cos(theta),
        c0[1] + radius * np.sin(theta),
        color="black",
        linewidth=1.2,
    )

    time_text = ax.text(0.02, 0.98, "", transform=ax.transAxes, va="top", ha="left")

    def init():
        n = idx[0]
        scat.set_offsets(x[n])
        c = xA * np.sin(float(omega) * float(t[n]))
        wall.set_data(
            c[0] + radius * np.cos(theta),
            c[1] + radius * np.sin(theta),
        )
        time_text.set_text(f"t = {float(t[n]):.3f}")
        return scat, wall, time_text

    def update(frame_i):
        n = idx[frame_i]
        scat.set_offsets(x[n])
        c = xA * np.sin(float(omega) * float(t[n]))
        wall.set_data(
            c[0] + radius * np.cos(theta),
            c[1] + radius * np.sin(theta),
        )
        time_text.set_text(f"t = {float(t[n]):.3f}")
        return scat, wall, time_text

    ani = FuncAnimation(
        fig,
        update,
        frames=len(idx),
        init_func=init,
        blit=True,
        interval=int(1000 / fps),
    )

    return ani

## Parameters

In [ ]:
N = 200

radius = np.float32(5.0)
dt = np.float32(1.0e-5)
T_max = np.float32(20.0)
save_every = 1000

# The neighbour-search box must contain the whole moving circle.
box = np.array([-6.0, 6.0, -6.0, 6.0], dtype=np.float32)

m = np.ones(N, dtype=np.float32)
rad = np.full(N, 0.12, dtype=np.float32)

group_mobile = np.array([0, N], dtype=np.int64)

k_contact = np.float32(1.0e4)
gamma_contact = np.float32(5.0)

# Define the oscillating circular wall.
xA = np.array([0., 0.5], dtype=np.float32)
omega = np.float32(4.0 * np.pi * 0.5)

k_w = np.float32(1.0e4)
gamma_w = np.float32(5.0)

g = np.array([0.0, -9.81], dtype=np.float32)

## Initial Condition

In [ ]:
rng = np.random.default_rng(1)

x0 = np.empty((N, 2), dtype=np.float32)

for i in range(N):
    r = (radius - rad[i] - 0.5) * np.sqrt(rng.random())
    th = 2.0 * np.pi * rng.random()

    x0[i, 0] = r * np.cos(th)
    x0[i, 1] = r * np.sin(th)

v0 = rng.normal(0.0, 0.5, size=(N, 2)).astype(np.float32)

## Euclidean Simulation

In [ ]:
tic = time.perf_counter()
t, x, v = esol.simulate_euclidean_particles(
    box,
    x0,
    v0,
    m,
    rad,
    dt,
    T_max,
    group_mobile,
    k_contact,
    gamma_contact,
    radius,
    xA,
    omega,
    k_w,
    gamma_w,
    g,
    save_every,
)
euclidean_time = time.perf_counter() - tic

print(f"Euclidean runtime: {euclidean_time:.3f} s")

## Euclidean Video

The wall is moving for visualization.

In [ ]:
VIDEO_DIR = VALIDATION_DIR / "Videos"
VIDEO_DIR.mkdir(exist_ok=True)
video_fps = 60

ani_cartesian = animate_euclidean_moving_circle_clean(
    t,
    x,
    radius,
    xA,
    omega,
    box=box,
    stride=5,
    s=25,
    fps=video_fps,
)

html_path = VIDEO_DIR / "cartesian.html"
html_path.write_text(ani_cartesian.to_jshtml(), encoding="utf-8")
print(f"saved HTML: {html_path}")

avi_path = VIDEO_DIR / "cartesian.avi"
if animation.writers.is_available("ffmpeg"):
    ani_cartesian.save(
        avi_path,
        writer=animation.FFMpegWriter(fps=video_fps, codec="mpeg4", bitrate=3000),
        dpi=150,
    )
    print(f"saved AVI: {avi_path}")
else:
    print("ffmpeg is not available, so the AVI file was not written. The HTML animation was saved.")

plt.close(ani_cartesian._fig)
HTML(html_path.read_text(encoding="utf-8"))


## Polar Simulation

In [ ]:
q0, p0 = psol.euclidean_to_polar_particles(x0, v0)

tic = time.perf_counter()
t_p, q_p, p_p, x_p, v_p = psol.simulate_polar_particles(
    box,
    q0,
    p0,
    m,
    rad,
    dt,
    T_max,
    group_mobile,
    k_contact,
    gamma_contact,
    radius,
    xA,
    omega,
    k_w,
    gamma_w,
    g,
    save_every,
)
polar_time = time.perf_counter() - tic

assert np.allclose(t, t_p)

print(f"Polar runtime: {polar_time:.3f} s")
print(f"Polar / Euclidean runtime: {polar_time / euclidean_time:.3f}")

## Polar-Coordinate Video

The particles are shown in wall-attached polar coordinates. The radial coordinate is normalized by the circle radius, so the wall is fixed at `r = 1`.


In [ ]:
def wall_attached_polar_coordinates(t, x, radius, xA, omega):
    xA = np.asarray(xA, dtype=float)
    q = np.empty_like(x, dtype=float)

    for n, tn in enumerate(t):
        centre = xA * np.sin(float(omega) * float(tn))
        rel = x[n] - centre[None, :]
        q[n, :, 0] = np.sqrt(rel[:, 0]**2 + rel[:, 1]**2) / float(radius)
        q[n, :, 1] = np.mod(np.arctan2(rel[:, 1], rel[:, 0]), 2.0 * np.pi)

    return q


def animate_particles_in_wall_attached_polar_plane(
    t,
    x,
    radius,
    xA,
    omega,
    fps=60,
    stride=1,
    s=20,
    wall_color="black",
):
    q_wall = wall_attached_polar_coordinates(t, x, radius, xA, omega)
    idx = np.arange(0, len(t), max(1, int(stride)), dtype=np.int64)

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.set_xlim(0.0, 2.0 * np.pi)
    ax.set_ylim(0.0, 1.1)
    ax.set_xlabel(r"$\theta$")
    ax.set_ylabel(r"$r/R$")

    scat = ax.scatter(q_wall[idx[0], :, 1], q_wall[idx[0], :, 0], s=s)
    wall, = ax.plot(
        [0.0, 2.0 * np.pi],
        [1.0, 1.0],
        color=wall_color,
        linewidth=1.2,
    )
    time_text = ax.text(0.02, 0.98, "", transform=ax.transAxes, va="top", ha="left")

    def init():
        n = idx[0]
        scat.set_offsets(np.column_stack((q_wall[n, :, 1], q_wall[n, :, 0])))
        time_text.set_text(f"t = {float(t[n]):.3f}")
        return scat, wall, time_text

    def update(frame_i):
        n = idx[frame_i]
        scat.set_offsets(np.column_stack((q_wall[n, :, 1], q_wall[n, :, 0])))
        time_text.set_text(f"t = {float(t[n]):.3f}")
        return scat, wall, time_text

    ani = FuncAnimation(
        fig,
        update,
        frames=len(idx),
        init_func=init,
        blit=True,
        interval=int(1000 / fps),
    )

    return ani

In [ ]:
q_wall_p = wall_attached_polar_coordinates(t_p, x_p, radius, xA, omega)

VIDEO_DIR = VALIDATION_DIR / "Videos"
VIDEO_DIR.mkdir(exist_ok=True)
video_fps = 60

ani_polar = animate_particles_in_wall_attached_polar_plane(
    t_p,
    x_p,
    radius,
    xA,
    omega,
    stride=5,
    s=25,
    fps=video_fps,
    wall_color="black",
)

html_path = VIDEO_DIR / "polar.html"
html_path.write_text(ani_polar.to_jshtml(), encoding="utf-8")
print(f"saved HTML: {html_path}")

avi_path = VIDEO_DIR / "polar.avi"
if animation.writers.is_available("ffmpeg"):
    ani_polar.save(
        avi_path,
        writer=animation.FFMpegWriter(fps=video_fps, codec="mpeg4", bitrate=3000),
        dpi=150,
    )
    print(f"saved AVI: {avi_path}")
else:
    print("ffmpeg is not available, so the AVI file was not written. The HTML animation was saved.")

plt.close(ani_polar._fig)
HTML(html_path.read_text(encoding="utf-8"))


## Static Figures

One static frame is saved for each solver view. The polar-coordinate snapshot uses the same wall-attached coordinates as the video, so the wall appears as the fixed straight boundary `r = 1`.


In [ ]:
def moving_circle_wall(radius, xA, omega, time_value, n=400):
    theta = np.linspace(0.0, 2.0 * np.pi, n)
    c = np.asarray(xA, dtype=float) * np.sin(float(omega) * float(time_value))
    wall = np.empty((n, 2), dtype=float)
    wall[:, 0] = c[0] + float(radius) * np.cos(theta)
    wall[:, 1] = c[1] + float(radius) * np.sin(theta)
    return wall


snapshot_id = len(t) - 30

wall = moving_circle_wall(radius, xA, omega, t[snapshot_id])
fig, ax = plt.subplots(figsize=(5, 5))
ax.plot(wall[:, 0], wall[:, 1], color="black", lw=1.2)
ax.scatter(x[snapshot_id, :, 0], x[snapshot_id, :, 1], s=12)
ax.set_aspect("equal", adjustable="box")
ax.set_xlim(box[0], box[1])
ax.set_ylim(box[2], box[3])
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title(f"Euclidean solver, t = {float(t[snapshot_id]):.3f}")
fig.savefig("multiparticle_euclidean_snapshot.png", dpi=300, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(7, 4))
ax.axhline(1.0, color="black", lw=1.2)
ax.scatter(q_wall_p[snapshot_id, :, 1], q_wall_p[snapshot_id, :, 0], s=12)
ax.set_xlim(0.0, 2.0 * np.pi)
ax.set_ylim(0.0, 1.1)
ax.set_xlabel(r"$\theta$")
ax.set_ylabel(r"$r/R$")
ax.set_title(f"Wall-attached polar solver, t = {float(t_p[snapshot_id]):.3f}")
fig.savefig("multiparticle_polar_snapshot.png", dpi=300, bbox_inches="tight")
plt.show()


## Kinetic Energy

For a many-particle system, individual trajectories are sensitive to small collision-timing differences. The total kinetic energy is the more useful comparison.

In [ ]:
K_e = 0.5 * np.sum(m[None, :] * np.sum(v**2, axis=2), axis=1)
K_p = 0.5 * np.sum(m[None, :] * np.sum(v_p**2, axis=2), axis=1)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(t, K_e, label="Euclidean")
ax.plot(t_p, K_p, "--", label="Polar mapped back")
ax.set_xlabel("t")
ax.set_ylabel("total kinetic energy")
ax.set_title("Time profile of total kinetic energy")
ax.grid(True)
ax.legend()
plt.show()

print(f"Initial Euclidean K: {K_e[0]:.6e}")
print(f"Initial polar K:     {K_p[0]:.6e}")
print(f"Max |K_e - K_p| / K_e(0): {np.max(np.abs(K_e - K_p)) / K_e[0]:.6e}")